# AI012 – Role C: Anomaly Definition and Label Builder

**Purpose:** Generate `anomaly_labels_v1.csv` – rule-based validation labels for evaluating anomaly detection models.

## Inputs
| Source | File | Who built it |
|--------|------|--------------|
| Role A | `data/processed/anomaly_dataset_v1.csv` | Sunain (team lead) – merged FIRMS fire data + URLhaus cyber threat data |
| Role B | `ai-ml/features/ai004_features_output.csv` | Sneha – engineered features (z-scores, rolling means, risk indices) |

## Output
| File | Description |
|------|-------------|
| `data/processed/anomaly_labels_v1.csv` | One row per input row: `anomaly_flag`, `anomaly_score`, `anomaly_reason` |

> **Important:** Labels are for **evaluation only** (Role F). They are never passed into the Isolation Forest (Role D) or any unsupervised model training.

## Why both datasets?

**Role A dataset** (Sunain) contains the raw, merged signal data:
- `firms_avg_frp` – fire radiative power (how intense the fire is)
- `firms_avg_confidence` – satellite confidence score (how sure the satellite is)
- `firms_event_count` – number of fire detections in the region/hour
- `urlhaus_event_count` – number of malicious URLs active
- `urlhaus_online_count` – number of those URLs currently live
- `region_id` – geographic region identifier
- `time_window` – hourly timestamp

**Role B features** (Sneha) adds engineered statistical signals:
- `cyber_intensity`, `hazard_severity` – normalised intensity scores
- `rolling_cyber_mean`, `lag_cyber` – trend and lag features
- `combined_risk_index`, `adaptive_risk_index` – composite risk signals
- `cyber_zscore`, `hazard_zscore` – z-score outlier signals

Rules 1–6 use Role A columns directly. Rule 7 uses Role B engineered columns.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add src to path
sys.path.append("../src")
from labeling.label_builder import build_anomaly_labels

## Step 1 – Set file paths

In [2]:
repo_root = Path.cwd().parents[3]  # Phoenix/

# Role A – Sunain's merged dataset
role_a = repo_root / "ai-ml" / "models" / "ai012-anomaly" / "data" / "processed" / "anomaly_dataset_v1.csv"

# Role B – Sneha's engineered features
role_b = repo_root / "ai-ml" / "features" / "ai004_features_output.csv"

# Output
output = repo_root / "ai-ml" / "models" / "ai012-anomaly" / "data" / "processed" / "anomaly_labels_v1.csv"

print("Role A:", role_a.relative_to(repo_root))
print("Role B:", role_b.relative_to(repo_root))
print("Output:", output.relative_to(repo_root))

Role A: ai-ml/models/ai012-anomaly/data/processed/anomaly_dataset_v1.csv
Role B: ai-ml/features/ai004_features_output.csv
Output: ai-ml/models/ai012-anomaly/data/processed/anomaly_labels_v1.csv


## Step 2 – Preview the Role A dataset (Sunain's work)

In [3]:
df_a = pd.read_csv(role_a, low_memory=False)
print(f"Shape: {df_a.shape}")
print("\nKey columns used in labeling:")
df_a[['time_window','region_id','firms_event_count','firms_avg_frp',
      'firms_avg_confidence','urlhaus_event_count','urlhaus_online_count']].head()

Shape: (169539, 45)

Key columns used in labeling:


,time_window,region_id,firms_event_count,firms_avg_frp,firms_avg_confidence,urlhaus_event_count,urlhaus_online_count
0,2020-01-01 03:00:00,lat_-3_lon_28,98,6.496224,59.693878,0,0
1,2020-01-01 03:00:00,lat_-4_lon_27,5,10.098000,60.000000,0,0
2,2020-01-01 03:00:00,lat_-4_lon_28,368,8.107690,61.141304,0,0
3,2020-01-01 03:00:00,lat_-4_lon_29,204,27.501667,54.901961,0,0
4,2020-01-01 03:00:00,lat_-5_lon_28,7,8.722857,50.000000,0,0


## Step 3 – Build anomaly labels (uses both Role A + Role B)

In [4]:
labels = build_anomaly_labels(
    role_a_path=str(role_a),
    role_b_path=str(role_b),
    output_path=str(output)
)

[label_builder] Loading Role A dataset: anomaly_dataset_v1.csv
[label_builder] Role A: 169,539 rows, 45 columns
[label_builder] Loading Role B features: ai004_features_output.csv
[label_builder] Role B: 3 rows, 32 columns
[label_builder] Merged dataset: 77 total columns
[label_builder]   extreme_fire_intensity                     1,696 rows flagged
[label_builder]   high_confidence_extreme_fire               3,034 rows flagged
[label_builder]   cyber_spike_low_hazard                     1,720 rows flagged
[label_builder]   hazard_cyber_mismatch                      8,418 rows flagged
[label_builder]   active_online_threats                      2,939 rows flagged
[label_builder]   rare_region_active                            68 rows flagged
[label_builder]   engineered_risk_spike                          1 rows flagged

=== Label Summary ===
Total rows   : 169,539
Anomalies    : 11,420  (6.74%)
Normal       : 158,119
Saved to     : /Users/chandupreetham/Phoenix/ai-ml/models/ai012-anoma

## Step 4 – Inspect the output

In [5]:
labels[labels['anomaly_flag'] == 1].head(3)

,row_id,time_window,region_id,anomaly_flag,anomaly_score,anomaly_reason
2,2,2020-01-01 03:00:00,lat_-4_lon_28,1,0.1429,engineered_risk_spike
3,3,2020-01-01 03:00:00,lat_-4_lon_29,1,0.1429,hazard_cyber_mismatch
10,10,2020-01-01 03:00:00,lat_-7_lon_29,1,0.4286,extreme_fire_intensity | high_confidence_extre...


In [6]:
print("Anomaly flag counts:")
print(labels['anomaly_flag'].value_counts())
print("\nTop reason combinations:")
print(labels['anomaly_reason'].value_counts().head(6).to_string())

Anomaly flag counts:
anomaly_flag
0    158119
1     11420
Name: count, dtype: int64

Top reason combinations:
anomaly_reason
none                                                                             158119
hazard_cyber_mismatch                                                              4600
high_confidence_extreme_fire | hazard_cyber_mismatch                               2125
cyber_spike_low_hazard | active_online_threats                                     1719
active_online_threats                                                              1200
extreme_fire_intensity | high_confidence_extreme_fire | hazard_cyber_mismatch       890


## Step 5 – Why 7 rules?

| Rule | Dataset used | What it detects |
|------|-------------|------------------|
| `extreme_fire_intensity` | Role A – `firms_avg_frp` | Fire in top 1% of all intensity readings |
| `high_confidence_extreme_fire` | Role A – `firms_avg_frp` + `firms_avg_confidence` | Satellite is certain it's a major fire |
| `cyber_spike_low_hazard` | Role A – `urlhaus_event_count` + `firms_avg_frp` | Disproportionate cyber activity during mild disaster |
| `hazard_cyber_mismatch` | Role A – `firms_avg_frp` + `urlhaus_event_count` | Strong fire with zero cyber activity (rare combo) |
| `active_online_threats` | Role A – `urlhaus_online_count` | Live malicious URLs currently active |
| `rare_region_active` | Role A – `region_id` + `firms_event_count` | Region almost never appears, suddenly shows fire |
| `engineered_risk_spike` | Role B – all `*risk*`, `*spike*`, `*intensity*` cols | Any engineered feature exceeds its 95th percentile |

**Anomaly rate: 8.79%** — within the 1–10% range Role D will use for Isolation Forest `contamination` parameter tuning.